# The Residual Stream

## What This Is
The residual stream is the core architectural insight of the transformer: each layer *adds* to a running sum (the residual stream) rather than replacing it. This means:

- Every layer has direct read/write access to the same stream
- Information from early layers is preserved throughout
- We can understand what each layer contributes by looking at its additive updates

## Logit Lens
The logit lens (Nostalgebraist, 2020) reads the predictions at each layer by projecting the intermediate residual stream through the unembedding matrix. This shows how the model's prediction develops over depth.

For a model predicting the next token:
- Early layers: often the prediction is poorly calibrated, looks like the input token
- Middle layers: semantic content begins to appear
- Later layers: final refinement of the prediction

In [ ]:
import numpy as np

np.random.seed(42)

# We need a model with trained weights for meaningful logit lens
# We'll demonstrate the concept with a pretrained-like setup

D_MODEL = 16
VOCAB_SIZE = 50
SEQ_LEN = 8
N_HEADS = 2
D_HEAD = D_MODEL // N_HEADS

class LogitLensModel:
    """Transformer with logit lens capability."""
    
    def __init__(self):
        np.random.seed(0)
        self.W_embed = np.random.randn(VOCAB_SIZE, D_MODEL) * 0.3
        self.W_pos = np.random.randn(SEQ_LEN, D_MODEL) * 0.1
        
        # Multi-layer weights
        self.layers = []
        for l in range(4):
            self.layers.append({
                'W_Q': [np.random.randn(D_MODEL, D_HEAD) * 0.1 for _ in range(N_HEADS)],
                'W_K': [np.random.randn(D_MODEL, D_HEAD) * 0.1 for _ in range(N_HEADS)],
                'W_V': [np.random.randn(D_MODEL, D_HEAD) * 0.1 for _ in range(N_HEADS)],
                'W_O': np.random.randn(N_HEADS * D_HEAD, D_MODEL) * 0.1,
                'W_mlp': np.random.randn(D_MODEL, 4 * D_MODEL) * 0.1,
                'W_mlp_out': np.random.randn(4 * D_MODEL, D_MODEL) * 0.1,
            })
        self.W_unembed = np.random.randn(D_MODEL, VOCAB_SIZE) * 0.1
        
        # Make unembedding somewhat aligned with embedding for interpretability
        self.W_unembed = self.W_embed.T * 0.5
        
        self.residuals = []  # track residual stream at each layer
    
    def softmax(self, x, axis=-1):
        ex = np.exp(x - x.max(axis=axis, keepdims=True))
        return ex / ex.sum(axis=axis, keepdims=True)
    
    def layer_norm(self, x):
        return (x - x.mean(-1, keepdims=True)) / (x.std(-1, keepdims=True) + 1e-5)
    
    def forward(self, tokens):
        T = len(tokens)
        x = self.W_embed[tokens] + self.W_pos[:T]
        self.residuals = [('embed', x.copy())]
        
        for l, layer in enumerate(self.layers):
            # Attention
            xn = self.layer_norm(x)
            heads = []
            for h in range(N_HEADS):
                Q = xn @ layer['W_Q'][h]; K = xn @ layer['W_K'][h]; V = xn @ layer['W_V'][h]
                scores = Q @ K.T / np.sqrt(D_HEAD)
                mask = np.triu(np.ones((T, T)) * -1e9, k=1)
                attn = self.softmax(scores + mask)
                heads.append(attn @ V)
            attn_out = np.concatenate(heads, axis=-1) @ layer['W_O']
            x = x + attn_out
            
            # MLP
            xn = self.layer_norm(x)
            x = x + np.maximum(0, xn @ layer['W_mlp']) @ layer['W_mlp_out']
            self.residuals.append((f'layer{l}', x.copy()))
        
        return x @ self.W_unembed
    
    def logit_lens(self, position: int):
        """Apply unembedding at each layer to see predictions develop."""
        results = []
        for name, resid in self.residuals:
            logits = resid[position] @ self.W_unembed
            top_token = np.argmax(logits)
            probs = self.softmax(logits)
            results.append({
                'layer': name,
                'top_token': top_token,
                'confidence': float(probs[top_token]),
                'entropy': float(-(probs * np.log(probs + 1e-10)).sum()),
            })
        return results

model = LogitLensModel()
tokens = [5, 12, 3, 8, 20, 7, 4]
logits = model.forward(tokens)

print('Logit Lens: Prediction at each layer (position 6)')
print(f'{'Layer':<12} {'Top Token':>10} {'Confidence':>12} {'Entropy':>10}')
print('-' * 50)
for r in model.logit_lens(position=6):
    print(f'{r["layer"]:<12} {r["top_token"]:>10} {r["confidence"]:>12.4f} {r["entropy"]:>10.4f}')
